# 📓 04. Activity Contributions

### 🧮 Objective: Calculate activity scores for Combat, Collection, and Exploration.


### 🧹 Output Artifacts:
- `../../data/processed/4_activity_contributions.csv`


In [1]:
import pandas as pd
import numpy as np

INPUT_PATH = '../../data/processed/3_normalized_telemetry.csv'
OUTPUT_PATH = '../../data/processed/4_activity_contributions.csv'

print("Libraries imported")

Libraries imported


In [2]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


## 🔥 CRITICAL BUG HISTORY: The 'Raw Sum' Disaster

> **The Bug:** Originally, the column selector used a loose filter that included both normalized data AND original `rawJson.*` columns.
>
> **The Math Error:**
> `score_explore = normalized_dist(0.5) + raw_dist(4500.0) = 4500.5`
> This massive raw value dwarfed Combat (0.5) and Collection (0.5).
>
> **The Consequence:** It appeared players ignored combat, when in reality the math was biased by measurement units (meters vs kills). We now strictly filter columns.



In [ ]:
# =============================================================================
# ACTIVITY SCORE CALCULATION — v2 (Revised Formula)
# =============================================================================
#
# CHANGES FROM v1:
# 1. AVERAGES instead of sums: each category is divided by its feature count.
#    This gives every archetype an equal ceiling of 1.0, regardless of how
#    many features it uses. Without this, Combat (4 features) had a structural
#    score advantage over Collection (3) and Exploration (previously 3, now 2).
#
# 2. timeOutOfCombat REMOVED from Exploration:
#    - It accumulated passively for any player not in combat — including
#      combat-intent players who simply had no enemies available to fight.
#    - It is the arithmetic complement of timeInCombat (they sum to session
#      duration), making them redundant and inversely correlated.
#    - Including it caused combat-seeking players on sparse maps to be
#      misclassified as Explorers.
#    - Exploration is now measured by ACTIVE signals only.
#
# FEATURE GROUPS:
#   Combat (4):    enemiesHit, damageDone, timeInCombat, kills
#   Collection (3): itemsCollected, pickupAttempts, timeNearInteractables
#   Exploration (2): distanceTraveled, timeSprinting
# =============================================================================

combat_features  = ['enemiesHit', 'damageDone', 'timeInCombat', 'kills']
collect_features = ['itemsCollected', 'pickupAttempts', 'timeNearInteractables']
explore_features = ['distanceTraveled', 'timeSprinting']  # timeOutOfCombat excluded

df['score_combat']  = df[combat_features].mean(axis=1)   # avg of 4, range [0, 1]
df['score_collect'] = df[collect_features].mean(axis=1)  # avg of 3, range [0, 1]
df['score_explore'] = df[explore_features].mean(axis=1)  # avg of 2, range [0, 1]

df['score_total'] = df['score_combat'] + df['score_collect'] + df['score_explore']

print("Calculated activity scores (v2 — averages, timeOutOfCombat excluded from explore)")
print(f"score_combat  range: [{df['score_combat'].min():.3f}, {df['score_combat'].max():.3f}]")
print(f"score_collect range: [{df['score_collect'].min():.3f}, {df['score_collect'].max():.3f}]")
print(f"score_explore range: [{df['score_explore'].min():.3f}, {df['score_explore'].max():.3f}]")

In [4]:
# Calculate percentages
df['pct_combat'] = df['score_combat'] / df['score_total'].replace(0, np.nan)
df['pct_collect'] = df['score_collect'] / df['score_total'].replace(0, np.nan)
df['pct_explore'] = df['score_explore'] / df['score_total'].replace(0, np.nan)

# Fill NaN (no activity) with equal distribution
df[['pct_combat', 'pct_collect', 'pct_explore']] = df[['pct_combat', 'pct_collect', 'pct_explore']].fillna(1/3)

print("Calculated activity percentages")

Calculated activity percentages


In [5]:
# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n Saved to {OUTPUT_PATH}")


 Saved to ../../data/processed/4_activity_contributions.csv
